In [1]:
import torch
from ultralytics import YOLO    
from repc3k2 import register_repc3k2_for_ultralytics

In [2]:
device = 0 if torch.cuda.is_available() else "cpu" 
print(f"Kullanilan cihaz: {'GPU (cuda:0)' if device == 0 else 'CPU'}")
if device == "cpu":
        print("UYARI: GPU bulunamadi, egitim CPU'da yapilacak ve cok yavas olabilir.")

Kullanilan cihaz: GPU (cuda:0)


In [3]:
register_repc3k2_for_ultralytics()        

repc3k2.RepC3k2

In [4]:
# 1) P2 head'li YOLO26 mimarisini (henüz eğitilmemiş yapı) oluştur
#    "s" ölçeğini baseline ile aynı tut ki ağırlık transferi daha sağlıklı olsun
model = YOLO("repc3k2-2.yaml").load("../baseline/runs/detect/train/weights/best.pt")  # kendi baseline yolunu yaz

# 2) Baseline eğitiminden çıkan ağırlıkları yükle
#    .load() ismi/shape'i eşleşen katmanları (backbone, P3-P4-P5) otomatik aktarır,
#    yeni eklenen P2 katmanları rastgele başlatılmış olarak kalır
#model.load("runs/detect/train/weights/best.pt")  # kendi baseline yolunu yaz

WARNING no model scale passed. Assuming scale='n'.
Transferred 348/831 items from pretrained weights


In [5]:
# --- Aşama A: Backbone dondurulmuş, yeni katmanlar + head ısınıyor ---
model.train(
    data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
    epochs=500,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    device=device,
    name="ablation_repc3k2",
)

Ultralytics 8.4.118  Python-3.12.9 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../../YOLO.v1i.yolo26/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=repc3k2-2.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ablation_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002349CBEE4B0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model_path = Path("runs/detect/ablation_repc3k2/weights/best.pt")  #TODO : kendi model yolunu yaz
if model_path.exists():
    model = YOLO(str(model_path))
    print(f"Test ediliyor: {model_path}")
    results = model.val(
        data="../../../dataset2/yolo26/YOLO.v1i.yolo26/data.yaml",
        split="test",
        imgsz=640,
        device=0 if torch.cuda.is_available() else "cpu",
    )

    metrics = results.results_dict
    precision = metrics.get("metrics/precision(B)", None)
    recall = metrics.get("metrics/recall(B)", None)
    map50 = metrics.get("metrics/mAP50(B)", None)
    map5095 = metrics.get("metrics/mAP50-95(B)", None)

    print("\nTest metrikleri:")
    print(f"Precision: {precision * 100:.2f}%" if precision is not None else "Precision: bulunamadı")
    print(f"Recall: {recall * 100:.2f}%" if recall is not None else "Recall: bulunamadı")
    print(f"mAP@0.5: {map50 * 100:.2f}%" if map50 is not None else "mAP@0.5: bulunamadı")
    print(f"mAP@0.5-0.95: {map5095 * 100:.2f}%" if map5095 is not None else "mAP@0.5-0.95: bulunamadı")
else:
    print(f"Model bulunamadı: {model_path}")


Test ediliyor: runs\detect\ablation_repc3k2-2\weights\best.pt
Ultralytics 8.4.118  Python-3.12.9 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
repc3k2-2 summary: 176 layers, 2,787,386 parameters, 0 gradients, 6.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 82.823.4 MB/s, size: 26.7 KB)
val: Scanning C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\labels.cache... 279 images, 0 backgrounds, 10 corrupt: 100% ━━━━━━━━━━━━ 279/279  0.0s
val: C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\images\im171_jpg.rf.e4ef6151dffe621e681fe4ca6f8e2f1b.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\images\im227_jpg.rf.5070aa166043d7744a73e2b8ff14b0b4.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: C:\Users\ertug\Desktop\yolo\YOLO.v1i.yolov11\test\images\im227_jpg.rf.88d4ecb5fc66b992377cec7e5d187b24.jpg: ignoring corrupt image/label: labels mix segment and detectio